# Fine-Tuning, LoRA & QLoRA — Google Colab Notebook

In [1]:
!nvidia-smi

Sat Aug 22 04:52:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Installing Required Libraries


In [2]:
%pip install -q -U \
  transformers \
  datasets \
  peft \
  bitsandbytes \
  accelerate \
  sentencepiece \
  protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.0 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.0 which is incompatible.


#Importing Libraries

In [3]:
import os
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. QLoRA training may not work properly on CPU.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


#Configuration

In [4]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

USE_QLORA = True

OUTPUT_DIR = "/content/lora_qlora_adapter"

MAX_LENGTH = 512

print("Base model:", BASE_MODEL)
print("Using QLoRA:", USE_QLORA)
print("Output directory:", OUTPUT_DIR)

Base model: Qwen/Qwen2.5-0.5B-Instruct
Using QLoRA: True
Output directory: /content/lora_qlora_adapter


#Creating Instructional Dataset

In [5]:
training_data = [
    {
        "instruction": "What is Generative AI?",
        "response": "Generative AI is a type of artificial intelligence that can create new content such as text, images, audio, video, and code."
    },
    {
        "instruction": "Explain RAG in simple words.",
        "response": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from documents and gives that context to a language model so it can answer more accurately."
    },
    {
        "instruction": "What is fine-tuning?",
        "response": "Fine-tuning is the process of taking a pre-trained model and training it further on a specific dataset so it performs better for a particular task or domain."
    },
    {
        "instruction": "Why is full fine-tuning expensive?",
        "response": "Full fine-tuning updates all model parameters, so it requires more GPU memory, more training time, more storage, and higher cost."
    },
    {
        "instruction": "What is LoRA?",
        "response": "LoRA, or Low-Rank Adaptation, is a parameter-efficient fine-tuning method that freezes the base model and trains only small adapter matrices."
    },
    {
        "instruction": "What is QLoRA?",
        "response": "QLoRA is Quantized LoRA. It loads the base model in 4-bit precision and trains LoRA adapters, reducing memory usage while keeping good performance."
    },
    {
        "instruction": "What is the main difference between LoRA and QLoRA?",
        "response": "LoRA trains small adapter matrices on a normal precision base model, while QLoRA uses a 4-bit quantized base model to save more memory."
    },
    {
        "instruction": "What is the rank r in LoRA?",
        "response": "The rank r controls the size of the low-rank adapter matrices. A higher rank gives more capacity but uses more memory and compute."
    },
    {
        "instruction": "What is LoRA alpha?",
        "response": "LoRA alpha is a scaling factor that controls the strength of the LoRA update added to the frozen base model."
    },
    {
        "instruction": "What is LoRA dropout?",
        "response": "LoRA dropout randomly disables some adapter activations during training to reduce overfitting."
    },
    {
        "instruction": "When should we use LoRA?",
        "response": "Use LoRA when you want to adapt a model efficiently without updating all base model parameters."
    },
    {
        "instruction": "When should we use QLoRA?",
        "response": "Use QLoRA when GPU memory is limited and you want to fine-tune a larger model using 4-bit quantization with LoRA adapters."
    },
    {
        "instruction": "What is PEFT?",
        "response": "PEFT stands for Parameter-Efficient Fine-Tuning. It means adapting a model by training only a small number of parameters instead of the full model."
    },
    {
        "instruction": "What are adapters in LoRA?",
        "response": "Adapters are small trainable components added to the model layers. In LoRA, they learn task-specific updates while the base model remains frozen."
    },
    {
        "instruction": "What is an instruction dataset?",
        "response": "An instruction dataset contains prompts or questions along with ideal responses. It teaches the model how to respond to specific instructions."
    },
    {
        "instruction": "Give one use case of fine-tuning.",
        "response": "A banking chatbot can be fine-tuned on banking FAQs, policies, and customer support examples to give more domain-specific responses."
    },
    {
        "instruction": "What is the benefit of saving only LoRA adapters?",
        "response": "LoRA adapters are small, so they are easier to store, share, and load compared with saving the full fine-tuned model."
    },
    {
        "instruction": "What is quantization?",
        "response": "Quantization reduces the precision of model weights, such as from 16-bit to 4-bit, to reduce memory usage and make the model easier to run."
    },
    {
        "instruction": "Why do we evaluate after fine-tuning?",
        "response": "Evaluation checks whether the fine-tuned model gives better, safer, and more accurate answers on the target task."
    },
    {
        "instruction": "What is a common mistake in fine-tuning?",
        "response": "A common mistake is using low-quality or irrelevant data. Fine-tuning quality depends heavily on the quality of the dataset."
    },
]

df = pd.DataFrame(training_data)
df.head()

,instruction,response
0,What is Generative AI?,Generative AI is a type of artificial intellig...
1,Explain RAG in simple words.,RAG stands for Retrieval-Augmented Generation....
2,What is fine-tuning?,Fine-tuning is the process of taking a pre-tra...
3,Why is full fine-tuning expensive?,"Full fine-tuning updates all model parameters,..."
4,What is LoRA?,"LoRA, or Low-Rank Adaptation, is a parameter-e..."


In [6]:
len(df)

20

#Converting DataFrame to Hugging Face Dataset

In [7]:
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.15,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 17
})
Dataset({
    features: ['instruction', 'response'],
    num_rows: 3
})


#Loading Tokenizer

The tokenizer converts text into tokens.

For chat models, we format examples as:

```text
system message
user instruction
assistant response
```


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>


#Format Dataset as Chat Examples

In [9]:
SYSTEM_MESSAGE = (
    "You are a helpful AI teacher. "
    "Answer clearly and simply for intermediate students."
)


def format_chat_example(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


formatted_train = train_dataset.map(format_chat_example)
formatted_eval = eval_dataset.map(format_chat_example)

print(formatted_train[0]["text"])

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

<|im_start|>system
You are a helpful AI teacher. Answer clearly and simply for intermediate students.<|im_end|>
<|im_start|>user
What is the rank r in LoRA?<|im_end|>
<|im_start|>assistant
The rank r controls the size of the low-rank adapter matrices. A higher rank gives more capacity but uses more memory and compute.<|im_end|>



#Tokenize Dataset


In [10]:
def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


tokenized_train = formatted_train.map(
    tokenize_function,
    batched=False,
    remove_columns=formatted_train.column_names
)

tokenized_eval = formatted_eval.map(
    tokenize_function,
    batched=False,
    remove_columns=formatted_eval.column_names
)

print(tokenized_train)
print(tokenized_eval)

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


#Loading Base Model

In [11]:
if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model = prepare_model_for_kbit_training(model)

else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

model.config.use_cache = False

print("Base model loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.


#Adding LoRA Adapters


In [12]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

print("LoRA adapters added.")
model.print_trainable_parameters()

LoRA adapters added.
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


#Training Arguments


In [13]:
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=10,
    save_total_limit=2,
    fp16=use_fp16,
    report_to="none",
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
)

print("Training arguments ready.")

Training arguments ready.


#Create Trainer

In [14]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("Trainer is ready.")

Trainer is ready.


#Fine-Tuning


In [15]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
5,3.307463,3.165813


TrainOutput(global_step=5, training_loss=4.442426872253418, metrics={'train_runtime': 9.333, 'train_samples_per_second': 1.821, 'train_steps_per_second': 0.536, 'total_flos': 19150348615680.0, 'train_loss': 4.442426872253418, 'epoch': 1.0})

# Save LoRA / QLoRA Adapter Creating Zip file


In [16]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Adapter saved at:", OUTPUT_DIR)

Adapter saved at: /content/lora_qlora_adapter


In [17]:
from pathlib import Path

adapter_path = Path("/content/lora_qlora_adapter")

if adapter_path.exists():
    print("Adapter folder found.")
    print("Files inside adapter folder:")
    for file in adapter_path.iterdir():
        print("-", file.name)
else:
    print("Adapter folder not found. Please run the training and save-adapter cells first.")

Adapter folder found.
Files inside adapter folder:
- tokenizer.json
- README.md
- adapter_config.json
- chat_template.jinja
- checkpoint-5
- tokenizer_config.json
- adapter_model.safetensors


In [18]:
import shutil

adapter_folder = "/content/lora_qlora_adapter"
zip_output = "/content/lora_qlora_adapter"

shutil.make_archive(
    base_name=zip_output,
    format="zip",
    root_dir=adapter_folder
)

print("Adapter zipped successfully:")
print("/content/lora_qlora_adapter.zip")

Adapter zipped successfully:
/content/lora_qlora_adapter.zip


In [19]:
from google.colab import files

files.download("/content/lora_qlora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Testing the Fine-Tuned Model


In [20]:
def generate_answer(question, max_new_tokens=180):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()


print(generate_answer("Explain QLoRA in simple words."))

QLoRA stands for "Quantized Long Range", which is an approach to improve the quality of training deep learning models by using quantization, where each element of the model's data is represented with fewer bits than the original.

Think of it like adding more precision to your calculator when you're trying to solve complex math problems. You start with lots of decimal points (more precision) but as you get closer to solving the equation, you use only a few decimal points (less precision). This helps create smoother results that are easier to interpret and apply.

In the context of machine learning, QLoRA can be used to:

1. Improve convergence speed: By reducing the number of bits per element, we can train models faster.
2. Reduce noise: Using less precision means that errors might not always show up as quickly.
3. Create better visualizations: The reduced bit depth allows for clearer


#Testing More Questions

In [21]:
questions = [
    "What is LoRA?",
    "What is the difference between LoRA and QLoRA?",
    "Why is full fine-tuning expensive?",
    "What is the role of rank r in LoRA?",
    "When should we use QLoRA?"
]

for question in questions:
    print("=" * 80)
    print("Question:", question)
    print("Answer:", generate_answer(question))
    print()

Question: What is LoRA?
Answer: LoRA stands for Long Short-Term Memory, which is a type of neural network designed to handle long sequences of data with short-term dependencies. It's commonly used in natural language processing tasks like translation, summarization, and text classification.

Question: What is the difference between LoRA and QLoRA?
Answer: Lora is a type of layer-wise recurrent learning, which is an improvement over LSTMs by allowing for more flexible attention patterns. It enables users to fine-tune layers in their own models rather than relying on pretrained models, making it useful for tasks like translation or visual question answering.

Question: Why is full fine-tuning expensive?
Answer: Fine-tuning involves training the model on a larger dataset than the original one, to improve its performance on a specific task. However, it requires more computational resources and time. The cost of fine-tuning depends on several factors such as the size of the model, the compl

In [22]:
# Example reload code.
# Run this in a fresh runtime after training if needed.

'''
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_PATH = "/content/lora_qlora_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
'''
print("Reload example is available in this cell.")

Reload example is available in this cell.
